# 🍪 Cookie Checker — Google Colab

Check cookies, extract stealer logs, and grab Netflix tokens — all on Google's fast servers.

### Commands available:
| Command | What it does |
|---|---|
| `/check` | Validate cookies (txt, json, zip) |
| `/log` | Extract cookies from stealer log archives |
| `/nftoken` | Get a Netflix login URL from cookies |
| `/services` | List all 40+ supported services |

> **⚡ Run Cell 1 first (Setup) — takes ~60 seconds, only needed once per session.**

In [ ]:
#@title ⚙️ **STEP 1 — Setup** (Run this first, once per session)
%%bash
set -e

echo '📦 Installing Node.js 20...'
curl -fsSL https://deb.nodesource.com/setup_20.x | sudo -E bash - > /dev/null 2>&1
sudo apt-get install -y nodejs > /dev/null 2>&1
echo "✅ Node.js $(node -v) ready"

echo '📦 Installing 7zip (for RAR/7z archives)...'
sudo apt-get install -y p7zip-full > /dev/null 2>&1
echo '✅ 7zip ready'

if [ -d '/content/shell-version-cookie-checker' ]; then
  echo '🔄 Updating repo...'
  cd /content/shell-version-cookie-checker
  git pull --quiet
else
  echo '📥 Cloning repo...'
  git clone --quiet https://github.com/geminineel/shell-version-cookie-checker.git /content/shell-version-cookie-checker
  cd /content/shell-version-cookie-checker
fi

echo '📦 Installing dependencies...'
npm install --silent 2>/dev/null

echo '🔨 Building...'
npm run build 2>/dev/null

echo ''
echo '✅ All done! Cookie Checker is ready.'
echo '👇 Scroll down and run the command cells below.'

---
## 📁 Upload Your File
Run the cell below to upload a file from your computer, **OR** skip it and paste a CDN/direct download URL directly into the command cells.

In [ ]:
#@title 📤 Upload a file (optional — skip if using a URL)
from google.colab import files
import os

print('Select your file (zip, rar, 7z, txt, json, cookies):')
uploaded = files.upload()

for filename in uploaded.keys():
    dest = f'/content/{filename}'
    with open(dest, 'wb') as f:
        f.write(uploaded[filename])
    size_mb = os.path.getsize(dest) / (1024*1024)
    print(f'✅ Saved: {dest}  ({size_mb:.1f} MB)')
    print(f'📋 Copy this path into the command cells: /content/{filename}')

---
## 🔍 `/check` — Validate Cookies

Checks if cookies are valid and which plan/tier they have.

```
/check <file_or_url> [service]
```

Examples:
- `/check /content/cookies.zip`
- `/check /content/netflix.txt netflix.com`
- `/check https://cdn.example.com/cookies.zip`

In [ ]:
#@title 🔍 /check — Validate Cookies
import subprocess, os

FILE = '/content/cookies.zip'  #@param {type:"string"}
SERVICE = ''  #@param {type:"string"} [description: "optional: e.g. netflix.com — leave blank to auto-detect"]

cmd = ['/check', FILE]
if SERVICE.strip():
    cmd.append(SERVICE.strip())

env = os.environ.copy()
env['NO_COLOR'] = '1'
env['FORCE_COLOR'] = '0'

print(f'Running: {" ".join(cmd)}')
print('─' * 50)

result = subprocess.run(
    ['node', 'dist/cli.js'] + cmd,
    cwd='/content/shell-version-cookie-checker',
    capture_output=True, text=True, env=env
)
print(result.stdout)
if result.stderr:
    print('[stderr]', result.stderr[:2000])

# List any output files created
out_dir = '/content/shell-version-cookie-checker/output'
if os.path.exists(out_dir):
    files_out = os.listdir(out_dir)
    if files_out:
        print('\n📦 Output files:')
        for f in files_out:
            size = os.path.getsize(f'{out_dir}/{f}') / 1024
            print(f'  {f}  ({size:.0f} KB)')

---
## 📂 `/log` — Extract Cookies from Stealer Logs

Pulls cookies for specific domains out of stealer log archives.

```
/log <file_or_url> <domain1> [domain2 ...]
```

Examples:
- `/log /content/logs.zip netflix.com spotify.com`
- `/log /content/logs.rar discord.com chatgpt.com`
- `/log https://cdn.example.com/logs.zip netflix.com`

In [ ]:
#@title 📂 /log — Extract from Stealer Logs
import subprocess, os

FILE = '/content/logs.zip'  #@param {type:"string"}
DOMAINS = 'netflix.com spotify.com'  #@param {type:"string"} [description: "space-separated domains"]

domains = DOMAINS.strip().split()
cmd = ['/log', FILE] + domains

env = os.environ.copy()
env['NO_COLOR'] = '1'
env['FORCE_COLOR'] = '0'

print(f'Running: {" ".join(cmd)}')
print('─' * 50)

result = subprocess.run(
    ['node', 'dist/cli.js'] + cmd,
    cwd='/content/shell-version-cookie-checker',
    capture_output=True, text=True, env=env
)
print(result.stdout)
if result.stderr:
    print('[stderr]', result.stderr[:2000])

# List output files
out_dir = '/content/shell-version-cookie-checker/output'
if os.path.exists(out_dir):
    files_out = os.listdir(out_dir)
    if files_out:
        print('\n📦 Output files:')
        for f in files_out:
            size = os.path.getsize(f'{out_dir}/{f}') / 1024
            print(f'  {f}  ({size:.0f} KB)')

---
## 🔑 `/nftoken` — Netflix Login URL

Reads the `NetflixId` cookie and calls the Netflix iOS API to generate a short-lived login URL (~1 hour).

```
/nftoken <file_or_url>
```

Examples:
- `/nftoken /content/netflix.txt`
- `/nftoken https://cdn.example.com/netflix.txt`

> Open the generated URL in **Chrome or Safari** — not in Telegram or any in-app browser.

In [ ]:
#@title 🔑 /nftoken — Netflix Login URL
import subprocess, os

FILE = '/content/netflix.txt'  #@param {type:"string"}

cmd = ['/nftoken', FILE]

env = os.environ.copy()
env['NO_COLOR'] = '1'
env['FORCE_COLOR'] = '0'

print(f'Running: {" ".join(cmd)}')
print('─' * 50)

result = subprocess.run(
    ['node', 'dist/cli.js'] + cmd,
    cwd='/content/shell-version-cookie-checker',
    capture_output=True, text=True, env=env
)
print(result.stdout)
if result.stderr:
    print('[stderr]', result.stderr[:2000])

---
## 📋 `/services` — All Supported Services

In [ ]:
#@title 📋 /services — List all supported services
import subprocess, os

env = os.environ.copy()
env['NO_COLOR'] = '1'
env['FORCE_COLOR'] = '0'

result = subprocess.run(
    ['node', 'dist/cli.js', '/services'],
    cwd='/content/shell-version-cookie-checker',
    capture_output=True, text=True, env=env
)
print(result.stdout)

---
## 💾 Download Output Files

Run this cell to download all result zip files to your computer.

In [ ]:
#@title 💾 Download all output files
from google.colab import files
import os, glob

out_dir = '/content/shell-version-cookie-checker/output'
output_files = glob.glob(f'{out_dir}/*')

if not output_files:
    print('No output files yet. Run /check or /log first.')
else:
    print(f'Found {len(output_files)} file(s):')
    for f in output_files:
        size_kb = os.path.getsize(f) / 1024
        print(f'  📦 {os.path.basename(f)}  ({size_kb:.0f} KB)')
    print('\nDownloading...')
    for f in output_files:
        files.download(f)
    print('✅ Done!')

---
## ⚡ Power User: Run Any Command Directly

Paste any command here exactly like you would in the terminal.

In [ ]:
#@title ⚡ Run any command
import subprocess, os

COMMAND = '/log /content/logs.zip netflix.com spotify.com'  #@param {type:"string"}

env = os.environ.copy()
env['NO_COLOR'] = '1'
env['FORCE_COLOR'] = '0'

args = COMMAND.strip().split()
print(f'Running: {COMMAND}')
print('─' * 50)

result = subprocess.run(
    ['node', 'dist/cli.js'] + args,
    cwd='/content/shell-version-cookie-checker',
    capture_output=True, text=True, env=env
)
print(result.stdout)
if result.stderr:
    print('[stderr]', result.stderr[:2000])